# C2 companion — Train your first neural network in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/course-curriculum/notebooks/c2-first-neural-net.ipynb)

In the chapter [](../content/2_c2-what-is-a-neural-net.md) we *built* a tiny network by hand and ran it with random weights. Here we **train** a real one in PyTorch on the same two-crescent `make_moons` data, watching the loss fall and the curved boundary snap into place. Run this on Colab with a GPU (Runtime → Change runtime type → GPU) — though this toy model trains fine on CPU too.

In [ ]:
# Install PyTorch and scikit-learn (Colab already has most of this).
!pip install -q torch scikit-learn matplotlib

In [ ]:
# GPU check: True means a GPU is available and we'll use it.
import torch
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

## 1. The same data as the chapter

We recreate the standardized two-crescent dataset and turn it into PyTorch **tensors** — the
array type PyTorch trains on. We move them onto the chosen device (GPU if present).

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

np.random.seed(0)
X, y = make_moons(n_samples=300, noise=0.2, random_state=0)
X = (X - X.mean(axis=0)) / X.std(axis=0)        # standardize, just like the chapter

X_t = torch.tensor(X, dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)
print("inputs:", X_t.shape, " labels:", y_t.shape)

## 2. Define the network

This is the same shape as the hand-built net in the chapter: two inputs → a hidden layer of
8 neurons with a bending activation → one output neuron. PyTorch creates and tracks all the
weights for us; `nn.Sigmoid` is the same squashing bend we wrote by hand.

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(2, 8),     # hidden layer: 2 inputs -> 8 neurons (weights + biases)
    nn.Tanh(),           # bending activation (a cousin of sigmoid, trains a bit faster)
    nn.Linear(8, 1),     # output neuron: 8 hidden signals -> 1 score
    nn.Sigmoid(),        # squash to a probability between 0 and 1
).to(device)

print(model)

## 3. The training loop

This is gradient descent (chapter [](../content/4_c4-how-models-learn.md)) in real code:

- **loss** = binary cross-entropy, "how wrong" the probabilities are;
- **optimizer** = the thing that takes the downhill step for every weight at once;
- each **epoch** = one pass over the data: predict → measure loss → step downhill.

PyTorch computes the downhill direction for us with `loss.backward()` — no hand-peeking
needed.

In [ ]:
loss_fn = nn.BCELoss()                                  # how-wrong score for probabilities
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)  # takes the downhill steps

losses = []
for epoch in range(300):
    optimizer.zero_grad()          # clear last step's downhill info
    preds = model(X_t)             # forward pass: predictions for every point
    loss = loss_fn(preds, y_t)     # how wrong are we?
    loss.backward()                # work out the downhill direction for every weight
    optimizer.step()               # take one downhill step
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"epoch {epoch:>3}  loss {loss.item():.3f}")

accuracy = ((model(X_t) >= 0.5).float() == y_t).float().mean().item()
print(f"\nFinal training accuracy: {accuracy:.1%}")

## 4. Watch the loss fall

The loss should slide downhill over the epochs — exactly the valley-descent picture from the
chapter, now with thousands of weights moving together.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(losses, color="#16a085")
plt.xlabel("epoch")
plt.ylabel("loss (how wrong)")
plt.title("Training loss rolling downhill")
plt.show()

## 5. See the learned curved boundary

Compare this to the chapter's *untrained* boundary: after training, the curve should hug the
gap between the two crescents almost perfectly.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2.5, 2.5, 300))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=device)
with torch.no_grad():
    zz = model(grid).cpu().numpy().reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=0.6)
for label, color in [(0, "#2980b9"), (1, "#c0392b")]:
    pts = X[y == label]
    plt.scatter(pts[:, 0], pts[:, 1], s=14, color=color,
                edgecolors="white", linewidths=0.3, label=f"class {label}")
plt.title("The trained network's curved boundary")
plt.legend()
plt.show()

## Your turn

1. **Shrink the network.** Change the hidden layer from 8 neurons to 2. Can it still curve
   tightly enough to separate the crescents? What does the final accuracy become?
2. **Change the learning rate.** Try `lr=0.001` (tiny steps) and `lr=1.0` (giant steps). How
   does each change the loss curve — does one crawl, and does one bounce?
3. **Harder data.** Raise `noise=0.2` to `noise=0.4` in `make_moons`. The crescents now
   overlap. What happens to accuracy, and why can't *any* model reach 100% here?